# --------------------------------------------------------------

In [22]:
# @title
# ============================================
# 🎮 WORDLE REGEX SOLVER - GOOGLE COLAB
# High Contrast & Readable Side Panels
# ============================================

import subprocess
import sys
import os

# ============================================
# CHECK AND INSTALL DEPENDENCIES (IF NEEDED)
# ============================================

def install_if_missing(package):
    """Install package only if not already installed"""
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing("ipywidgets")

# ============================================
# CHECK AND MOUNT GOOGLE DRIVE (IF NEEDED)
# ============================================

def check_and_mount_drive():
    """Mount Google Drive only if not already mounted"""
    drive_path = '/content/drive'

    if not os.path.exists(drive_path):
        try:
            from google.colab import drive
            drive.mount(drive_path)
        except Exception as e:
            pass

check_and_mount_drive()

# ============================================
# IMPORT LIBRARIES
# ============================================

from ipywidgets import *
import re

# ============================================
# LOAD WORD LIST FROM DRIVE
# ============================================

WORD_FILE_PATH = '/content/drive/path/to/WORDLE.txt'

def load_words():
    """Load words from the WORDLE.txt file"""
    words = []
    try:
        if not os.path.exists(WORD_FILE_PATH):
            return []

        with open(WORD_FILE_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip().upper()
                if len(word) == 5 and word.isalpha():
                    words.append(word)
        return words
    except Exception as e:
        return []

SAMPLE_WORDS = load_words()

if not SAMPLE_WORDS:
    SAMPLE_WORDS = ['ENTER', 'PANEL', 'SPILL', 'STILL', 'SWILL', 'TWILL', 'ALIEN', 'ALONE']

# ============================================
# SOLVER STATE & LOGIC
# ============================================

class WORDLESolver:
    def __init__(self, words):
        self.words = words
        self.reset()

    def reset(self):
        self.excluded = set()
        self.included = set()
        self.confirmed = {}
        self.excluded_at_pos = {}
        self.history = []
        self.step = 1

    def process_guess(self, word, colors):
        """Process user's guess and color feedback"""
        word = word.upper()

        for i, color in colors.items():
            i = int(i)
            letter = word[i]

            if color == 'grey':
                self.excluded.add(letter)
            elif color == 'yellow':
                self.included.add(letter)
                if i not in self.excluded_at_pos:
                    self.excluded_at_pos[i] = set()
                self.excluded_at_pos[i].add(letter)
            elif color == 'green':
                self.confirmed[str(i)] = letter
                self.included.add(letter)

    def generate_regex(self):
        """Generate regex pattern from current constraints"""
        pattern = []

        for pos in range(5):
            if str(pos) in self.confirmed:
                pattern.append(self.confirmed[str(pos)])
            else:
                excluded_here = self.excluded.copy()
                if pos in self.excluded_at_pos:
                    excluded_here.update(self.excluded_at_pos[pos])

                if excluded_here:
                    excluded_str = ''.join(sorted(excluded_here))
                    pattern.append(f"[^{excluded_str}]")
                else:
                    pattern.append('.')

        return '^' + ''.join(pattern) + '$'

    def get_candidates(self, regex):
        """Get matching words from WORDLE.txt"""
        compiled_regex = re.compile(regex)
        candidates = [w for w in self.words
                     if compiled_regex.match(w) and
                     all(letter in w for letter in self.included)]
        return sorted(candidates)

    def submit_guess(self, word, colors):
        """Submit a guess and get results"""
        self.process_guess(word, colors)
        regex = self.generate_regex()
        candidates = self.get_candidates(regex)

        self.history.append({
            'step': self.step,
            'word': word.upper(),
            'regex': regex,
            'count': len(candidates)
        })

        self.step += 1

        return {
            'regex': regex,
            'candidates': candidates,
            'history': self.history,
            'excluded': list(self.excluded),
            'included': list(self.included),
            'confirmed': self.confirmed,
            'excluded_at_pos': {str(k): list(v) for k, v in self.excluded_at_pos.items()}
        }

solver = WORDLESolver(SAMPLE_WORDS)

# ============================================
# GLOBAL STATE VARIABLES
# ============================================

color_selections = {}
letter_buttons = {}
current_step = 1
current_word = ""

# ============================================
# CREATE SIDE PANELS WITH HIGH CONTRAST
# ============================================

# Regex Summary Panel (Right Side) - HIGH CONTRAST
regex_panel = HTML('''
<div style="background: #1f2937; padding: 20px; border-radius: 15px; color: #f9fafb; height: 500px; overflow-y: auto;
            box-shadow: 0 4px 20px rgba(0,0,0,0.4); border: 2px solid #374151;">
    <h3 style="margin-top: 0; text-align: center; font-size: 1.4em; color: #ffffff; font-weight: bold;">🎯 Regex Summary</h3>
    <div id="regex-content">
        <p style="text-align: center; color: #d1d5db; margin-top: 50px; font-size: 1.1em;">
            📝 Regex patterns will appear here<br>after each guess
        </p>
    </div>
</div>
''')

# Word Suggestions Panel (Left Side) - HIGH CONTRAST
suggestions_panel = HTML('''
<div style="background: #0f172a; padding: 20px; border-radius: 15px; color: #f1f5f9; height: 500px; overflow-y: auto;
            box-shadow: 0 4px 20px rgba(0,0,0,0.4); border: 2px solid #1e293b;">
    <h3 style="margin-top: 0; text-align: center; font-size: 1.4em; color: #ffffff; font-weight: bold;">💡 Word Suggestions</h3>
    <div id="suggestions-content">
        <p style="text-align: center; color: #cbd5e1; margin-top: 50px; font-size: 1.1em;">
            🔤 Suggested words will appear here<br>after each guess
        </p>
    </div>
</div>
''')

# ============================================
# MAIN UI COMPONENTS
# ============================================

# Main title - HIGH CONTRAST
title = HTML(f'''
<div style="text-align: center; margin-bottom: 25px; background: #1e40af;
            padding: 20px; border-radius: 15px; color: white; box-shadow: 0 4px 15px rgba(0,0,0,0.3);
            border: 2px solid #3b82f6;">
    <h1 style="margin: 0 0 10px 0; font-size: 2.3em; color: #ffffff; font-weight: bold;">🎮 WORDLE Regex Solver</h1>
    <p style="margin: 5px 0; font-size: 1.2em; color: #dbeafe;">Step: <span id="step-counter">1</span></p>
    <p style="margin: 5px 0; font-size: 1.0em; color: #bfdbfe;">📚 Dictionary: {len(SAMPLE_WORDS)} words loaded</p>
</div>
''')

# Instructions - HIGH CONTRAST
instructions = HTML('''
<div style="background: #ffffff; padding: 20px; border-radius: 12px;
            margin: 15px 0; border: 2px solid #1e40af; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
    <h3 style="color: #1e40af; margin-top: 0; font-size: 1.3em; font-weight: bold;">📖 How to Use:</h3>
    <ol style="color: #1f2937; line-height: 1.8; padding-left: 20px; font-weight: 500; font-size: 1.0em;">
        <li>Enter your <strong style="color: #000000;">5-letter guessed word</strong> (e.g., "ENTER")</li>
        <li>A <strong style="color: #000000;">3×5 grid</strong> will appear with your word's letters in each row</li>
        <li>Click <strong style="color: #000000;">ONE letter per column</strong> to select the color for each position</li>
        <li>Click <strong style="color: #000000;">"Submit Guess"</strong> to see results in side panels</li>
    </ol>
</div>
''')

# Word input
word_input = Text(
    value='',
    placeholder='Type your 5-letter guess (e.g., ENTER)...',
    description='Guess:',
    style={'description_width': '70px'},
    layout=Layout(
        width='450px',
        height='50px',
        margin='15px auto',
        border='2px solid #1e40af',
        border_radius='8px'
    )
)

word_input_container = HBox([word_input], layout=Layout(justify_content='center'))

# Grid container
grid_container = VBox([])

# Buttons
submit_btn = Button(
    description='📤 Submit Guess',
    button_style='info',
    disabled=True,
    layout=Layout(width='180px', height='50px', margin='5px')
)

reset_btn = Button(
    description='🔄 Reset Grid',
    button_style='warning',
    layout=Layout(width='150px', height='50px', margin='5px')
)

clear_btn = Button(
    description='🗑️ Clear All',
    button_style='danger',
    layout=Layout(width='140px', height='50px', margin='5px')
)

# ============================================
# DYNAMIC GRID GENERATION
# ============================================

def create_letter_grid(word):
    """Create 3×5 grid with the input word's letters"""
    global letter_buttons, color_selections

    word = word.upper()
    color_selections = {}
    letter_buttons = {}

    rows = []

    row_configs = [
        {
            'color': 'grey',
            'bg_color': '#4b5563',
            'text_color': 'white',
            'label': '❌ Grey (Not in word)'
        },
        {
            'color': 'yellow',
            'bg_color': '#d97706',
            'text_color': 'white',
            'label': '🟨 Yellow (Wrong position)'
        },
        {
            'color': 'green',
            'bg_color': '#059669',
            'text_color': 'white',
            'label': '🟩 Green (Correct position)'
        }
    ]

    for row_idx, config in enumerate(row_configs):
        row_label = HTML(f"""
            <h4 style='color: {config['bg_color']}; margin: 25px 0 15px 0; text-align: center;
                       font-size: 1.2em; font-weight: bold;'>
                {config['label']}
            </h4>
        """)

        buttons = []
        for pos in range(5):
            letter = word[pos]

            btn = Button(
                description=f'{letter}',
                layout=Layout(
                    width='75px',
                    height='75px',
                    margin='8px',
                    border_radius='10px'
                ),
                style=dict(
                    button_color=config['bg_color'],
                    text_color=config['text_color'],
                    font_weight='bold',
                    font_size='20px'
                )
            )

            key = (row_idx, pos)
            letter_buttons[key] = {
                'button': btn,
                'letter': letter,
                'position': pos,
                'color': config['color'],
                'bg_color': config['bg_color'],
                'text_color': config['text_color'],
                'selected': False
            }

            btn.on_click(lambda b, r=row_idx, p=pos: on_letter_button_click(r, p))
            buttons.append(btn)

        button_row = HBox(
            buttons,
            layout=Layout(
                justify_content='center',
                padding='10px 0'
            )
        )

        rows.append(row_label)
        rows.append(button_row)

    return VBox(rows, layout=Layout(
        border='2px solid #cbd5e1',
        border_radius='15px',
        padding='25px',
        margin='20px auto',
        background_color='#ffffff',
        box_shadow='0 4px 15px rgba(0,0,0,0.1)',
        max_width='600px'
    ))

def on_letter_button_click(row, pos):
    """Handle letter button click"""
    global color_selections, letter_buttons

    clicked_button_data = letter_buttons[(row, pos)]
    color = clicked_button_data['color']
    position = clicked_button_data['position']

    # Clear previous selection for this position
    for r in range(3):
        key = (r, pos)
        if key in letter_buttons:
            btn_data = letter_buttons[key]
            btn = btn_data['button']

            if r == row:
                if btn_data['selected']:
                    btn_data['selected'] = False
                    reset_button_style(btn, btn_data)
                    if position in color_selections:
                        del color_selections[position]
                else:
                    btn_data['selected'] = True
                    btn.style = dict(
                        button_color='#1e40af',  # Strong blue
                        text_color='white',
                        font_weight='bold',
                        font_size='20px'
                    )
                    btn.button_style = ''
                    color_selections[position] = color
            else:
                if btn_data['selected']:
                    btn_data['selected'] = False
                    reset_button_style(btn, btn_data)

    check_submit_ready()

def reset_button_style(btn, btn_data):
    """Reset button to original color"""
    btn.style = dict(
        button_color=btn_data['bg_color'],
        text_color=btn_data['text_color'],
        font_weight='bold',
        font_size='20px'
    )
    btn.button_style = ''

def check_submit_ready():
    """Check if submit button should be enabled"""
    ready = len(color_selections) == 5
    submit_btn.disabled = not ready

    if ready:
        submit_btn.button_style = 'success'
        submit_btn.description = '✅ Ready to Submit!'
    else:
        submit_btn.button_style = 'info'
        submit_btn.description = f'📤 Submit Guess ({len(color_selections)}/5)'

# ============================================
# UPDATE SIDE PANELS WITH HIGH CONTRAST
# ============================================

def update_regex_panel(history):
    """Update the regex summary panel with high contrast colorful boxes"""
    regex_content = "<h3 style='margin-top: 0; text-align: center; color: #ffffff; font-weight: bold; font-size: 1.4em;'>🎯 Regex Evolution</h3>"

    # High contrast color palette
    colors = [
        {'bg': '#dc2626', 'text': '#ffffff'},  # Red
        {'bg': '#ea580c', 'text': '#ffffff'},  # Orange
        {'bg': '#ca8a04', 'text': '#ffffff'},  # Amber
        {'bg': '#059669', 'text': '#ffffff'},  # Green
        {'bg': '#0284c7', 'text': '#ffffff'},  # Sky
        {'bg': '#7c3aed', 'text': '#ffffff'},  # Violet
        {'bg': '#db2777', 'text': '#ffffff'},  # Pink
    ]

    for i, item in enumerate(history):
        color = colors[i % len(colors)]
        regex_content += f'''
        <div style="background: {color['bg']}; margin: 15px 0; padding: 18px; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(0,0,0,0.4); border: 2px solid rgba(255,255,255,0.2);">
            <div style="font-weight: bold; font-size: 1.2em; margin-bottom: 10px; color: {color['text']};">
                ✏️ Step {item['step']}: {item['word']}
            </div>
            <div style="font-family: 'Courier New', monospace; background: rgba(0,0,0,0.4);
                       padding: 12px; border-radius: 8px; font-size: 1.0em; margin: 10px 0;
                       color: {color['text']}; border: 1px solid rgba(255,255,255,0.3); font-weight: bold;">
                patt{item['step']} = r"{item['regex']}"
            </div>
            <div style="font-size: 1.0em; color: {color['text']}; font-weight: bold;">
                ✓ Matches: <strong>{item['count']} words</strong>
            </div>
        </div>
        '''

    regex_panel.value = f'''
    <div style="background: #1f2937; padding: 20px; border-radius: 15px; color: #f9fafb; height: 500px; overflow-y: auto;
                box-shadow: 0 4px 20px rgba(0,0,0,0.4); border: 2px solid #374151;">
        {regex_content}
    </div>
    '''

def update_suggestions_panel(candidates):
    """Update the suggestions panel with high contrast word list"""
    if not candidates:
        content = '''
        <h3 style='margin-top: 0; text-align: center; color: #ffffff; font-weight: bold; font-size: 1.4em;'>💡 No Matches</h3>
        <div style="text-align: center; color: #cbd5e1; margin-top: 50px; font-size: 1.1em;">
            ❌ No words match current pattern<br>
            Try different color combinations
        </div>
        '''
    elif len(candidates) == 1:
        content = f'''
        <h3 style='margin-top: 0; text-align: center; color: #ffffff; font-weight: bold; font-size: 1.4em;'>🎉 SOLUTION!</h3>
        <div style="text-align: center; margin-top: 50px;">
            <div style="background: #059669; padding: 25px; border-radius: 15px;
                       font-size: 2.2em; font-weight: bold; border: 3px solid #10b981;
                       color: white; box-shadow: 0 4px 15px rgba(0,0,0,0.3);">
                {candidates[0]}
            </div>
        </div>
        '''
    else:
        # Group words in rows of 3 for better display
        word_rows = []
        for i in range(0, min(len(candidates), 60), 3):
            row_words = candidates[i:i+3]
            word_boxes = []
            for word in row_words:
                word_boxes.append(f'''
                <div style="background: rgba(255,255,255,0.15); padding: 12px; margin: 6px;
                           border-radius: 10px; font-weight: bold; text-align: center;
                           border: 2px solid rgba(255,255,255,0.3); color: #ffffff; font-size: 1.0em;">
                    {word}
                </div>
                ''')
            word_rows.append(f'<div style="display: flex; justify-content: space-between;">{"".join(word_boxes)}</div>')

        remaining = len(candidates) - 60
        remaining_text = f"<p style='text-align: center; color: #cbd5e1; margin-top: 15px; font-weight: bold; font-size: 1.0em;'>... and {remaining} more words</p>" if remaining > 0 else ""

        content = f'''
        <h3 style='margin-top: 0; text-align: center; color: #ffffff; font-weight: bold; font-size: 1.4em;'>💡 {len(candidates)} Suggestions</h3>
        <div style="margin-top: 20px;">
            {"".join(word_rows)}
            {remaining_text}
        </div>
        '''

    suggestions_panel.value = f'''
    <div style="background: #0f172a; padding: 20px; border-radius: 15px; color: #f1f5f9; height: 500px; overflow-y: auto;
                box-shadow: 0 4px 20px rgba(0,0,0,0.4); border: 2px solid #1e293b;">
        {content}
    </div>
    '''

# ============================================
# EVENT HANDLERS
# ============================================

def on_word_input_change(change):
    """Handle word input change"""
    global current_word

    word = change['new'].upper().strip()

    if word and not word.isalpha():
        word = ''.join(c for c in word if c.isalpha())
        word_input.value = word
        return

    if len(word) > 5:
        word_input.value = word[:5]
        return

    if len(word) == 5:
        current_word = word
        grid = create_letter_grid(word)
        grid_container.children = [grid]
        submit_btn.disabled = True
        submit_btn.button_style = 'info'
        submit_btn.description = '📤 Submit Guess (0/5)'
    else:
        grid_container.children = []
        submit_btn.disabled = True
        submit_btn.button_style = 'info'
        submit_btn.description = '📤 Submit Guess'
        current_word = ""

word_input.observe(on_word_input_change, names='value')

def on_submit_click(b):
    """Handle submit button click"""
    global current_step, current_word

    word = current_word

    if len(word) != 5 or len(color_selections) != 5:
        return

    # Submit guess
    result = solver.submit_guess(word, color_selections)
    current_step = solver.step

    # Update side panels
    update_regex_panel(result['history'])
    update_suggestions_panel(result['candidates'])

    # Clear input and grid
    word_input.value = ''
    grid_container.children = []
    color_selections.clear()
    current_word = ""
    submit_btn.disabled = True
    submit_btn.button_style = 'info'
    submit_btn.description = '📤 Submit Guess'

    # Update step counter
    title.value = f'''
<div style="text-align: center; margin-bottom: 25px; background: #1e40af;
            padding: 20px; border-radius: 15px; color: white; box-shadow: 0 4px 15px rgba(0,0,0,0.3);
            border: 2px solid #3b82f6;">
    <h1 style="margin: 0 0 10px 0; font-size: 2.3em; color: #ffffff; font-weight: bold;">🎮 WORDLE Regex Solver</h1>
    <p style="margin: 5px 0; font-size: 1.2em; color: #dbeafe;">Step: <span id="step-counter">{current_step}</span></p>
    <p style="margin: 5px 0; font-size: 1.0em; color: #bfdbfe;">📚 Dictionary: {len(SAMPLE_WORDS)} words loaded</p>
</div>
'''

def on_reset_click(b):
    """Handle reset button click"""
    global color_selections, letter_buttons

    color_selections.clear()

    for key, data in letter_buttons.items():
        data['selected'] = False
        reset_button_style(data['button'], data)

    submit_btn.disabled = True
    submit_btn.button_style = 'info'
    submit_btn.description = '📤 Submit Guess (0/5)'

def on_clear_click(b):
    """Handle clear all button click"""
    global current_step, color_selections, current_word

    solver.reset()
    color_selections.clear()
    word_input.value = ''
    grid_container.children = []
    current_word = ""
    current_step = 1
    submit_btn.disabled = True
    submit_btn.button_style = 'info'
    submit_btn.description = '📤 Submit Guess'

    # Reset side panels
    regex_panel.value = '''
    <div style="background: #1f2937; padding: 20px; border-radius: 15px; color: #f9fafb; height: 500px; overflow-y: auto;
                box-shadow: 0 4px 20px rgba(0,0,0,0.4); border: 2px solid #374151;">
        <h3 style="margin-top: 0; text-align: center; font-size: 1.4em; color: #ffffff; font-weight: bold;">🎯 Regex Summary</h3>
        <div id="regex-content">
            <p style="text-align: center; color: #d1d5db; margin-top: 50px; font-size: 1.1em;">
                📝 Regex patterns will appear here<br>after each guess
            </p>
        </div>
    </div>
    '''

    suggestions_panel.value = '''
    <div style="background: #0f172a; padding: 20px; border-radius: 15px; color: #f1f5f9; height: 500px; overflow-y: auto;
                box-shadow: 0 4px 20px rgba(0,0,0,0.4); border: 2px solid #1e293b;">
        <h3 style="margin-top: 0; text-align: center; font-size: 1.4em; color: #ffffff; font-weight: bold;">💡 Word Suggestions</h3>
        <div id="suggestions-content">
            <p style="text-align: center; color: #cbd5e1; margin-top: 50px; font-size: 1.1em;">
                🔤 Suggested words will appear here<br>after each guess
            </p>
        </div>
    </div>
    '''

    title.value = f'''
<div style="text-align: center; margin-bottom: 25px; background: #1e40af;
            padding: 20px; border-radius: 15px; color: white; box-shadow: 0 4px 15px rgba(0,0,0,0.3);
            border: 2px solid #3b82f6;">
    <h1 style="margin: 0 0 10px 0; font-size: 2.3em; color: #ffffff; font-weight: bold;">🎮 WORDLE Regex Solver</h1>
    <p style="margin: 5px 0; font-size: 1.2em; color: #dbeafe;">Step: <span id="step-counter">1</span></p>
    <p style="margin: 5px 0; font-size: 1.0em; color: #bfdbfe;">📚 Dictionary: {len(SAMPLE_WORDS)} words loaded</p>
</div>
'''

# Attach event handlers
submit_btn.on_click(on_submit_click)
reset_btn.on_click(on_reset_click)
clear_btn.on_click(on_clear_click)

# ============================================
# MAIN LAYOUT WITH SIDE PANELS
# ============================================

# Center section (main interface)
center_section = VBox([
    word_input_container,
    grid_container,
    HBox([submit_btn, reset_btn, clear_btn], layout=Layout(
        justify_content='center',
        margin='25px 0',
        padding='10px',
        background_color='#f1f5f9',
        border_radius='10px'
    ))
], layout=Layout(
    padding='20px',
    flex='1 1 auto',
    max_width='600px'
))

# Main content area (3-column layout)
main_content = HBox([
    suggestions_panel,  # Left panel
    center_section,     # Center panel
    regex_panel        # Right panel
], layout=Layout(
    padding='0px 20px',
    margin='20px 0'
))

# Complete UI
ui = VBox([
    title,
    instructions,
    main_content
], layout=Layout(
    padding='25px',
    border='2px solid #e0e6ff',
    border_radius='15px',
    width='100%',
    background_color='#ffffff',
    box_shadow='0 4px 20px rgba(0,0,0,0.08)'
))

display(ui)
